In [35]:
import requests
import os
import re
import json

# Load from environment
api_key = os.environ.get("AZURE_OPENAI_KEY", "").strip('"')
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "").strip('"')

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

print("✅ Setup complete!")
print(f"Endpoint loaded: {endpoint[:50]}...")

✅ Setup complete!
Endpoint loaded: https://sudha-mmkhx2sq-eastus2.cognitiveservices.a...


In [36]:
api_key = "9BsEyRppMAcQpCBiC2Et1uGAY44JJIyfLg0oZElEppdZvqOuGUTQJQQJ99CCACHYHv6XJ3w3AAAAACOGBgtd"

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

print("✅ Key updated!")

✅ Key updated!


In [37]:
# Our document - imagine this is a PDF or company document
document = """
Azure is Microsoft's cloud computing platform. It offers over 200 services 
including virtual machines, databases, AI services, and storage solutions.

Azure Blob Storage is used to store unstructured data like images, videos, 
and documents. It is highly scalable and cost-effective.

Azure OpenAI Service provides access to GPT models like GPT-4o-mini. 
These models can understand and generate human-like text.

RAG stands for Retrieval Augmented Generation. It combines search with 
AI generation to answer questions from documents accurately.

Machine learning on Azure can be done using Azure Machine Learning service.
It supports training, deploying, and monitoring ML models at scale.
"""

print("✅ Document loaded!")
print(f"Total characters: {len(document)}")

✅ Document loaded!
Total characters: 694


In [38]:
def split_into_chunks(text,chunk_size=150):
    words=text.split()
    chunks=[]
    current_chunk=[]
    current_size=0
    
    for word in words:
        current_chunk.append(word)
        current_size +=len(word)
        
        if current_size>=chunk_size:
            chunks.append(" ".join(current_chunk))
            current_chunk=[]
            current_size=0
            
    if current_chunk:
        chunks.append(" ".join(current_chunk))
        
    return chunks
chunks = split_into_chunks(document)
print(f"😍 Document split into {len(chunks)} chunks!")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: {chunk}")

😍 Document split into 4 chunks!

Chunk 1: Azure is Microsoft's cloud computing platform. It offers over 200 services including virtual machines, databases, AI services, and storage solutions. Azure Blob Storage is used

Chunk 2: to store unstructured data like images, videos, and documents. It is highly scalable and cost-effective. Azure OpenAI Service provides access to GPT models like GPT-4o-mini. These

Chunk 3: models can understand and generate human-like text. RAG stands for Retrieval Augmented Generation. It combines search with AI generation to answer questions from documents accurately.

Chunk 4: Machine learning on Azure can be done using Azure Machine Learning service. It supports training, deploying, and monitoring ML models at scale.


In [39]:
def find_top_chunks(question, chunks, top=2):
    stopwords = ["what", "is", "the", "how", "are", "does", "this", "that"]
    
    question_words = [
        word.strip("?.,!") 
        for word in question.lower().split()
        if len(word) >= 3 and word not in stopwords
    ]
    
    scores = []
    for chunk in chunks:
        chunk_lower = chunk.lower()
        score = sum(1 for word in question_words 
                   if re.search(r'\b' + word + r'\b', chunk_lower))
        scores.append(score)
    
    # Get top 2 chunk indexes
    top_indexes = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top]
    top_chunks = " ".join([chunks[i] for i in top_indexes])
    return top_chunks

In [40]:
def ask_gpt_with_context(question, context):
    body = {
        "messages": [
            {"role": "system", "content": "Answer the question using only the context provided. If the answer is not in the context, say 'I don't know'."},
            {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
        ]
    }
    response = requests.post(endpoint, headers=headers, json=body)
    result = response.json()
    return result["choices"][0]["message"]["content"]

# Full RAG pipeline!
question = "What is RAG?"
relevant_chunk = find_relevant_chunk(question, chunks)
answer = ask_gpt_with_context(question, relevant_chunk)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Searching for keywords: ['rag']
Scores: [0, 0, 1, 0]
Question: What is RAG?

Answer: RAG stands for Retrieval Augmented Generation. It combines search with AI generation to answer questions from documents accurately.


In [44]:
print(ask_gpt_with_context("What is Azure Blob Storage?", 
      find_top_chunks("What is Azure Blob Storage?", chunks)))

print("---")

print(ask_gpt_with_context("What is Azure OpenAI?", 
      find_top_chunks("What is Azure OpenAI?", chunks)))

Azure Blob Storage is a service used to store unstructured data like images, videos, and documents. It is highly scalable and cost-effective.
---
Azure OpenAI Service provides access to GPT models like GPT-4o-mini.
